MEMO Debugging

In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [2]:
import torch
from MeMoHF.modelling_memo_tokenizer import MeMoTokenizer
from MeMoHF.modelling_memo_configuration import MeMoConfig
from MeMoHF.modelling_memo import MeMoForCausalLM
from MeMoHF.evaluating_memo import Evaluation
from MeMoHF.utils import seed_everything

seed_everything(42)

# d, h, l = 1024, 4, 4
# chunk_length = 256

# d, h, l = 1024, 4, 2
# chunk_length = 16
d,h,l = 2048, 4, 2
chunk_length =  (h ** l)

compositionOp='SCC'#'ProdWithShuffling' #'JLT' #


padding_vector_component_values= {
    'ProdWithShuffling': 1/(d**(1/2)),
    'Prod': 1/(d**(1/2)), 
    'Sum': 0,
    'SCC': 'circular_convolution_identity',
    'JLT': 'in_distribution'
}
padding_vector_component_values = padding_vector_component_values[compositionOp]


# Initializing a standard Tokenizer
max_length = chunk_length 
tokenizer = MeMoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b", 
                                          padding_side='left', truncation_side='left', 
                                          model_max_length=max_length, head_number=h)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.pad_token_id

# Intializing Memo Configuration
config = MeMoConfig(vocab_size=len(tokenizer), #tokenizer.vocab_size, 
    hidden_size=d, 
    num_hidden_layers=l,
    num_attention_heads=h,
    chunk_length=chunk_length,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    compositionOp=compositionOp,#'ProdWithShuffling',
    padding_seq_idx=tokenizer.pad_token_id,
    padding_vector_component_values=padding_vector_component_values #new!
                    
)

# Initializing the Memo Model from the configuration

model = MeMoForCausalLM(config) 
model.training = True
model.train()

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    model.to('cuda')


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPTNeoXTokenizer'. 
The class this function is called from is 'MeMoTokenizer'.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Setting pad token and pad token id = <|endoftext|>, 0
MeMoEmbedding circular_convolution_identity
MeMoEmbedding 0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  circular_convolution_identity
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  circular_convolution_identity
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
GPU: NVIDIA RTX A6000 is available.


In [3]:
model.memo.output_encoder.forward(torch.tensor([model.memo.output_encoder.padding_idx]))

tensor([[0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0',
       grad_fn=<EmbeddingBackward0>)

In [4]:
display(model.memo.encoder.weight[model.memo.encoder.padding_seq_idx], 
model.memo.output_encoder.weight[model.memo.encoder.padding_seq_idx])

tensor([1., 0., 0.,  ..., 0., 0., 0.], device='cuda:0',
       grad_fn=<SelectBackward0>)

tensor([0., 0., 0.,  ..., 0., 0., 0.], device='cuda:0',
       grad_fn=<SelectBackward0>)

In [5]:
model.memo.encoder.weight[model.memo.encoder.padding_idx], model.memo.output_encoder.weight[model.memo.encoder.padding_idx]

(tensor([1., 0., 0.,  ..., 0., 0., 0.], device='cuda:0',
        grad_fn=<SelectBackward0>),
 tensor([0., 0., 0.,  ..., 0., 0., 0.], device='cuda:0',
        grad_fn=<SelectBackward0>))

In [6]:
model

MeMoForCausalLM(
  (memo): MeMo(
    (encoder): MeMoEmbedding(50277, 2048, padding_idx=0)
    (output_encoder): MeMoEmbedding(50277, 2048, padding_idx=0)
    (layers): MeMoLayers(
      (0-1): 2 x MeMoLayer(
        (CMM_OUT): CorrelationMatrixMemory(in_features=2048, out_features=2048)
      )
    )
  )
  (lm_head): MeMoEmbedding(50277, 2048, padding_idx=0)
)

In [7]:
#my_first_text = "b c d e f a b c d e f"
my_first_text = "a b c d e f g h i l m"
my_second_text = "j k m a b c" # "n o p q r s t"#
memo_input_1 = tokenizer.get_text_batch_encoding([my_first_text]*1)  # Writing the same doc 8 times to stress the memorization with batch
memo_input_2 = tokenizer.get_text_batch_encoding([my_second_text]*1) # Writing the same doc 8 times to stress the memorization with batch

print(tokenizer.decode(memo_input_1['input_ids'][0]))
print(tokenizer.decode(memo_input_1['labels'][0]))

print(memo_input_1['input_ids'], len(memo_input_1['input_ids'][0]))
print(memo_input_1['labels'], len(memo_input_1['input_ids'][0]))

<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>a b c d e f g h i l
<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>a b c d e f g h i l m
tensor([[  0,   0,   0,   0,   0,   0,  66, 270, 260, 277, 299, 269, 305, 288,
         891, 298]]) 16
tensor([[  0,   0,   0,   0,   0,  66, 270, 260, 277, 299, 269, 305, 288, 891,
         298, 278]]) 16


In [8]:
memo_input_1

{'input_ids': tensor([[  0,   0,   0,   0,   0,   0,  66, 270, 260, 277, 299, 269, 305, 288,
          891, 298]]),
 'labels': tensor([[  0,   0,   0,   0,   0,  66, 270, 260, 277, 299, 269, 305, 288, 891,
          298, 278]])}

In [9]:
model.memo.encoder.padding_seq_idx, model.memo.encoder.padding_idx

(0, 0)

In [10]:
enc = model.memo.encoder.encode(memo_input_1['input_ids'])
dec = model.memo.encoder.decode(enc)
print(dec)

tokenizer.batch_decode(dec[0])

(tensor([[  0,   0,   0,   0,   0,   0,  66, 270, 260, 277, 299, 269, 305, 288,
         891, 298]], device='cuda:0'), tensor([[1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 0.9775, 1.0243, 0.9695,
         0.9847, 0.9980, 1.0480, 1.0281, 0.9932, 0.9709, 1.0278]],
       device='cuda:0', grad_fn=<MaxBackward0>))


['<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>a b c d e f g h i l']

In [11]:
model.memo.encoder.weight[270].T @ model.memo.encoder.weight[270] 

/tmp/ipykernel_2873057/3403464107.py:1: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3697.)
  model.memo.encoder.weight[270].T @ model.memo.encoder.weight[270]


tensor(1.0243, device='cuda:0', grad_fn=<DotBackward0>)

In [12]:
a = tokenizer.get_text_batch_encoding_for_loss(text=[my_first_text])
for k in a:
    print(k, a[k].shape)
    print(a[k])

input_ids torch.Size([1, 16])
tensor([[  0,   0,   0,   0,   0,   0,  66, 270, 260, 277, 299, 269, 305, 288,
         891, 298]])
labels torch.Size([1, 16])
tensor([[-100, -100, -100, -100, -100,   66,  270,  260,  277,  299,  269,  305,
          288,  891,  298,  278]])


In [13]:
# evaluation method
#from MeMoHF.evaluating_memo import EvaluationUpdateNew
#evaluation = EvaluationUpdateNew()
def perform_evaluation(model=None, tokenizer=None, text=None, truncation=None, starting_point=0,
                       decoders=None, topk=(1, 5, 10, 100), keep_token_stats=False):
    """
    decoders         : which Bayes decoders to score (None = all; "legacy" is always included)
    topk             : the k of the top-k accuracies
    keep_token_stats : keep the per-token counts in decoder_scores (large; they are always
                       available in token_accuracy['decoders'][name]['token_stats'])
 
    Returns
      out_loss              legacy loss (sum over valid tokens), as before
      token_accuracy        the full accuracy_results dict, as before (now with a 'decoders' entry)
      decoder_scores        {decoder: {avg_nll, ppl, avg_entropy, avg_confidence, accuracy,
                                       topk_acc: {k: acc}, num_tokens}}
      decoder_scores_flat   same numbers as one flat dict, keys like "bayes_T_prior/top10",
                            "fit_T/avg_nll" (convenient for logging / DataFrames)
    """
    model.eval()
    try:
        batch_inputs = tokenizer.get_text_batch_encoding_for_loss(text=text, truncation=truncation) #for_loss
        #print(batch_inputs)
        with torch.no_grad():
            actual_output = model.forward_with_loss_simple( 
                batch_inputs=batch_inputs,
                compute_accuracy=True,
                return_dict=True,
                starting_point=starting_point,
                decoders=decoders,
                topk=topk,
            )
 
            outs, pretokenized_score, _ = actual_output
 
        if outs is None:  # starting_point >= sequence length: nothing to score
            return dict(out_loss=None, token_accuracy=None, decoder_scores={}, decoder_scores_flat={})
 
        loss = outs['loss']
        del outs
    finally:
        model.train()
        torch.cuda.empty_cache()
 
    decoder_scores = {
        name: {k: v for k, v in res.items() if keep_token_stats or k != 'token_stats'}
        for name, res in pretokenized_score.get('decoders', {}).items()
    }
 
    decoder_scores_flat = {}
    for name, res in decoder_scores.items():
        for metric, value in res.items():
            if metric == 'token_stats':
                continue
            if metric == 'topk_acc':
                for k, acc in value.items():
                    decoder_scores_flat[f"{name}/top{k}"] = acc
            else:
                decoder_scores_flat[f"{name}/{metric}"] = value
 
    return dict(
        out_loss=loss,
        token_accuracy=pretokenized_score,
        decoder_scores=decoder_scores,
        decoder_scores_flat=decoder_scores_flat,
    )
 
 
def format_decoder_scores(decoder_scores):
    """Small text table of perform_evaluation(...)['decoder_scores'], sorted by NLL."""
    if not decoder_scores:
        return "(no decoder scores)"
    ks = sorted(next(iter(decoder_scores.values()))['topk_acc'])
    header = f"{'decoder':<15}{'nll':>8}{'ppl':>10}{'entropy':>9}{'conf':>7}" + "".join(f"{'top'+str(k):>8}" for k in ks)
    lines = [header]
    for name, r in sorted(decoder_scores.items(), key=lambda kv: kv[1]['avg_nll']):
        lines.append(f"{name:<15}{r['avg_nll']:8.3f}{r['ppl']:10.2f}{r['avg_entropy']:9.3f}{r['avg_confidence']:7.3f}"
                     + "".join(f"{r['topk_acc'][k]:8.3f}" for k in ks))
    return "\n".join(lines)

In [14]:
memo_input_1#['input_ids'].shape

{'input_ids': tensor([[  0,   0,   0,   0,   0,   0,  66, 270, 260, 277, 299, 269, 305, 288,
          891, 298]]),
 'labels': tensor([[  0,   0,   0,   0,   0,  66, 270, 260, 277, 299, 269, 305, 288, 891,
          298, 278]])}

In [15]:
# memorize input
# model.memorize_text(memo_input)
model.memorize_text(memo_input_1)

In [16]:
tokenizer.get_text_batch_encoding_for_loss([my_first_text]*1)#['input_ids'].shape

{'input_ids': tensor([[  0,   0,   0,   0,   0,   0,  66, 270, 260, 277, 299, 269, 305, 288,
          891, 298]]),
 'labels': tensor([[-100, -100, -100, -100, -100,   66,  270,  260,  277,  299,  269,  305,
           288,  891,  298,  278]])}

In [17]:
memo_input_1['labels'], [ 66,  66,  66,  66,  66,  66, 270, 260, 277, 299, 269, 305, 288, 891,
         298, 278]

(tensor([[  0,   0,   0,   0,   0,  66, 270, 260, 277, 299, 269, 305, 288, 891,
          298, 278]]),
 [66, 66, 66, 66, 66, 66, 270, 260, 277, 299, 269, 305, 288, 891, 298, 278])

In [18]:
model.memo.encoder.weight[tokenizer.pad_token_id], model.memo.output_encoder.weight[tokenizer.pad_token_id]

(tensor([1., 0., 0.,  ..., 0., 0., 0.], device='cuda:0',
        grad_fn=<SelectBackward0>),
 tensor([0., 0., 0.,  ..., 0., 0., 0.], device='cuda:0',
        grad_fn=<SelectBackward0>))

In [19]:
## Legacy decoder
o = model.retrieve(memo_input_1['input_ids'])
print(o.logits.argmax(dim=-1))#.shape


o.logits.max(dim=-1).indices == memo_input_1['labels'].to('cuda'), memo_input_1['labels']

tensor([[ 66,  66,  66,  66,  66,  66, 270, 260, 277, 299, 269, 305, 288, 891,
         298, 278]], device='cuda:0')


(tensor([[False, False, False, False, False,  True,  True,  True,  True,  True,
           True,  True,  True,  True,  True,  True]], device='cuda:0'),
 tensor([[  0,   0,   0,   0,   0,  66, 270, 260, 277, 299, 269, 305, 288, 891,
          298, 278]]))

In [20]:
for d in o.decoder_probs:
    print(d, o.decoder_probs[d].max(dim=-1).indices == memo_input_1['labels'].to('cuda'), o.decoder_probs[d].max(dim=-1).values)
    print()

legacy tensor([[False, False, False, False, False,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True]], device='cuda:0') tensor([[3.1549e-05, 3.1549e-05, 3.1549e-05, 3.1549e-05, 5.2703e-05, 5.2703e-05,
         5.2690e-05, 5.3784e-05, 5.3667e-05, 5.3820e-05, 5.4723e-05, 5.3800e-05,
         5.3437e-05, 5.3538e-05, 5.4019e-05, 5.2538e-05]], device='cuda:0',
       grad_fn=<MaxBackward0>)

dot_T1 tensor([[False, False, False, False, False,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True]], device='cuda:0') tensor([[3.1306e-05, 3.1306e-05, 3.1306e-05, 3.1306e-05, 5.1871e-05, 5.1871e-05,
         5.1563e-05, 5.3937e-05, 5.3550e-05, 5.3794e-05, 5.5598e-05, 5.3707e-05,
         5.3205e-05, 5.3391e-05, 5.4391e-05, 5.1346e-05]], device='cuda:0',
       grad_fn=<MaxBackward0>)

bayes_T tensor([[ True,  True,  True,  True, False,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True]], device='

In [21]:
results_1 = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=[my_first_text]*1
)
print(format_decoder_scores(results_1['decoder_scores']))
print('--------')
results_2 = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=[my_second_text]*1
)
print(format_decoder_scores(results_2['decoder_scores']))

decoder             nll       ppl  entropy   conf    top1    top5   top10  top100
bayes_T           0.000      1.00    0.000  1.000   1.000   1.000   1.000   1.000
bayes_T_prior     0.000      1.00    0.000  1.000   1.000   1.000   1.000   1.000
fit_T             0.000      1.00    0.000  1.000   1.000   1.000   1.000   1.000
fit_T_prior       0.000      1.00    0.000  1.000   1.000   1.000   1.000   1.000
legacy            9.836  18685.96   10.825  0.000   1.000   1.000   1.000   1.000
dot_T1            9.840  18764.92   10.825  0.000   1.000   1.000   1.000   1.000
--------
decoder             nll       ppl  entropy   conf    top1    top5   top10  top100
legacy           10.726  45535.51   10.825  0.000   0.000   0.333   0.500   0.500
dot_T1           10.820  50011.36   10.825  0.000   0.000   0.333   0.333   0.500
bayes_T        203032.422       inf    0.000  1.000   0.000   0.000   0.000   0.167
fit_T          203032.422       inf    0.000  1.000   0.000   0.000   0.000   0.167
bay

In [22]:
model.memorize_text(memo_input_2)

In [23]:
results_1 = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=[my_first_text]*1
)
print(format_decoder_scores(results_1['decoder_scores']))
print('--------')
results_2 = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=[my_second_text]*1
)
print(format_decoder_scores(results_2['decoder_scores']))

decoder             nll       ppl  entropy   conf    top1    top5   top10  top100
bayes_T           0.000      1.00    0.000  1.000   1.000   1.000   1.000   1.000
bayes_T_prior     0.000      1.00    0.000  1.000   1.000   1.000   1.000   1.000
fit_T             0.000      1.00    0.000  1.000   1.000   1.000   1.000   1.000
fit_T_prior       0.000      1.00    0.000  1.000   1.000   1.000   1.000   1.000
dot_T1            9.837  18710.92   10.825  0.000   1.000   1.000   1.000   1.000
legacy            9.849  18944.52   10.825  0.000   1.000   1.000   1.000   1.000
--------
decoder             nll       ppl  entropy   conf    top1    top5   top10  top100
bayes_T           0.000      1.00    0.000  1.000   1.000   1.000   1.000   1.000
bayes_T_prior     0.000      1.00    0.000  1.000   1.000   1.000   1.000   1.000
fit_T             0.000      1.00    0.000  1.000   1.000   1.000   1.000   1.000
fit_T_prior       0.000      1.00    0.000  1.000   1.000   1.000   1.000   1.000
dot_T1 

In [24]:
memo_input_2['labels']

tensor([[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  75, 465, 278, 247,
         270, 260]])

In [25]:
seed_everything(42)

# d, h, l = 1024, 4, 4
# chunk_length = 256

# d, h, l = 1024, 4, 2
# chunk_length = 16
d,h,l = 2048, 4, 3 #### increased #num_layers or chunk lenght --> lower performance
chunk_length = h ** l 

# Initializing a standard Tokenizer
max_length = chunk_length 
tokenizer = MeMoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b", 
                                          padding_side='left', truncation_side='left', 
                                          model_max_length=max_length, head_number=h)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.pad_token_id



# Intializing Memo Configuration
config = MeMoConfig(vocab_size=len(tokenizer), #tokenizer.vocab_size, 
    hidden_size=d, 
    num_hidden_layers=l,
    num_attention_heads=h,
    chunk_length=chunk_length,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    compositionOp=compositionOp,#'ProdWithShuffling',
    padding_seq_idx=tokenizer.pad_token_id,
    padding_vector_component_values=padding_vector_component_values  
                    
)

# Initializing the Memo Model from the configuration

model = MeMoForCausalLM(config) 
model.training = True
model.train()

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    model.to('cuda')


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPTNeoXTokenizer'. 
The class this function is called from is 'MeMoTokenizer'.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Setting pad token and pad token id = <|endoftext|>, 0
MeMoEmbedding circular_convolution_identity
MeMoEmbedding 0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  circular_convolution_identity
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  circular_convolution_identity
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
GPU: NVIDIA RTX A6000 is available.


In [26]:
from datasets import Dataset, DatasetDict, Features, Value, load_dataset, load_from_disk, concatenate_datasets

data = load_from_disk('original_training_data/new_sample/mini/n=000020')
data

Dataset({
    features: ['text'],
    num_rows: 20
})

In [27]:
memo_input_3 = tokenizer.get_text_batch_encoding([data[0]['text']]*1)
model.memorize_text(memo_input_3)

Token indices sequence length is longer than the specified maximum sequence length for this model (233 > 65). Running this sequence through the model will result in indexing errors


In [28]:
## too much padding added??
a = tokenizer.get_text_batch_encoding_for_loss([data[1]['text']])['input_ids']
a, a.shape

(tensor([[    0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,    40, 14399,  1321, 36029,   313,  6448,   818,
           4223,  8441,    10,   310,   247,  5112,  5702, 31926,   665,  7120,
            323,  4009,  2083,  1914,  5210, 16004,    13,   327, 10119,   432,
           6365,    91,    13,   347,   247,   259,  4940,    15,   187,   187,
           4941,   187,   187,  1413,    27, 14316,   952,   187,  1413,    27,
          12487, 10782,   187,  1413],
         [43421,   432, 19548,  4415,   313,  7955,  2688,    10,   187,  1413,
             27,  8836,   432,   596,   279,  6169,   187,  1413,    27, 28521,
           5842, 12916,   398,   187,  1413,    27, 21258, 23236,   187,  1413,
             27,  6739,  6365,    91,  3773,   187,  1413,    27,    53,  2108,
          16004,  3773,   187,  1413,    27,  2838,  2083,  1914,  5210, 16004,
           3773,   187,  1413,    27,    45, 14406,   337,  3773,   187,  1413,
 

In [29]:
memo_input_4 = tokenizer.get_text_batch_encoding([data[1]['text']]*1)
model.memorize_text(memo_input_4)

In [30]:
print("After memorizing 3 and 4")

results_1 = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=[my_first_text]*1
)

print(format_decoder_scores(results_1['decoder_scores']))
print("*"*80)

results_2 = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=[data[0]['text']]*1
)
print(format_decoder_scores(results_2['decoder_scores']))
print("*"*80)

results_3 = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=[data[1]['text']]*1
)
print(format_decoder_scores(results_3['decoder_scores']))
print("*"*80)


After memorizing 3 and 4
decoder             nll       ppl  entropy   conf    top1    top5   top10  top100
dot_T1           10.822  50102.49   10.825  0.000   0.000   0.000   0.000   0.000
legacy           10.825  50285.68   10.825  0.000   0.000   0.000   0.000   0.000
bayes_T        6860.928       inf    0.000  1.000   0.000   0.000   0.000   0.000
fit_T          6860.928       inf    0.000  1.000   0.000   0.000   0.000   0.000
bayes_T_prior  6863.804       inf    0.000  1.000   0.000   0.000   0.000   0.000
fit_T_prior    6863.804       inf    0.000  1.000   0.000   0.000   0.000   0.000
********************************************************************************


decoder             nll       ppl  entropy   conf    top1    top5   top10  top100
dot_T1            9.892  19775.51   10.825  0.000   0.996   1.000   1.000   1.000
legacy            9.975  21484.54   10.825  0.000   0.996   1.000   1.000   1.000
bayes_T          80.234       inf    0.000  1.000   0.964   1.000   1.000   1.000
fit_T            80.234       inf    0.000  1.000   0.964   1.000   1.000   1.000
bayes_T_prior    80.276       inf    0.000  1.000   0.964   1.000   1.000   1.000
fit_T_prior      80.276       inf    0.000  1.000   0.964   1.000   1.000   1.000
********************************************************************************
decoder             nll       ppl  entropy   conf    top1    top5   top10  top100
dot_T1            9.877  19476.29   10.825  0.000   1.000   1.000   1.000   1.000
legacy            9.964  21239.10   10.825  0.000   1.000   1.000   1.000   1.000
bayes_T          61.258401765843174257080999608320.00    0.000  1.000   0.982   1.000   1.000   1.0

In [31]:
seed_everything(42)

# d, h, l = 1024, 4, 4
# chunk_length = 256

# d, h, l = 1024, 4, 2
# chunk_length = 16
d,h,l = 2048, 4, 5 #### increased #num_layers or chunk lenght --> lower performance
chunk_length = h ** l 

# Initializing a standard Tokenizer
max_length = chunk_length 
tokenizer = MeMoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b", 
                                          padding_side='left', truncation_side='left', 
                                          model_max_length=max_length, head_number=h)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.pad_token_id



# Intializing Memo Configuration
config = MeMoConfig(vocab_size=len(tokenizer), #tokenizer.vocab_size, 
    hidden_size=d, 
    num_hidden_layers=l,
    num_attention_heads=h,
    chunk_length=chunk_length,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    compositionOp=compositionOp,
    padding_seq_idx=tokenizer.pad_token_id,
    padding_vector_component_values= padding_vector_component_values
                    
)


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPTNeoXTokenizer'. 
The class this function is called from is 'MeMoTokenizer'.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Setting pad token and pad token id = <|endoftext|>, 0


In [32]:
from datasets import Dataset, DatasetDict, Features, Value, load_dataset, load_from_disk, concatenate_datasets

data = load_from_disk('LearningEvaluation/training_data/scaling_curve/n=000050')
data

Dataset({
    features: ['text'],
    num_rows: 50
})

In [33]:
tok_data = data.map(lambda x: tokenizer.get_text_batch_encoding(x['text'], truncation=True))
tok_data.set_format(type='torch', columns=['input_ids', 'labels'])
tok_data

Dataset({
    features: ['text', 'input_ids', 'labels'],
    num_rows: 50
})

In [34]:
tok_data[0]['input_ids'].shape, tok_data[0]['labels'].shape

(torch.Size([1, 1024]), torch.Size([1, 1024]))

In [35]:
# Initializing the Memo Model from the configuration
import pandas as pd

model = MeMoForCausalLM(config) 
model.to('cuda')



for i in range(5):
    table = {}
    print(f"Memorizing doc {i+1}")
    model.memorize_text(tok_data[i])
    
    print(f"Retrieving doc {i+1}")  
    outputs = model.memo.retrieve(tok_data[i]['input_ids'])
    
    raw_recall = outputs['raw_recall_unpacked']                       # (l, B, T, d)

    DECODER_NAMES = ("dot_T1", "bayes_T", "bayes_T_prior", "fit_T", "fit_T_prior")

    E = model._candidate_matrix()
    E_sq = E.pow(2).sum(-1)
    
    for layer_level in range(l):
        table[layer_level] = {}
        per_layer = model._layer_decoder_logits(
            model.memo.layers[layer_level], raw_recall[layer_level], 
            E, E_sq, DECODER_NAMES)
        for d in per_layer.keys():
            per_layer[d] = per_layer[d].to('cpu')
            table[layer_level][d] = (tok_data[i]["labels"] == per_layer[d].max(dim=-1).indices).sum().item() / len(tok_data[i]["labels"][0])
            

    display(pd.DataFrame(table))

model = model.to('cpu')
del model
torch.cuda.empty_cache()

MeMoEmbedding circular_convolution_identity
MeMoEmbedding 0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  circular_convolution_identity


MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  circular_convolution_identity
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
Memorizing doc 1
Retrieving doc 1


,0,1,2,3,4
dot_T1,0.954102,0.997070,0.987305,0.939453,0.757812
bayes_T,0.951172,0.995117,0.984375,0.936523,0.749023
bayes_T_prior,0.951172,0.995117,0.984375,0.936523,0.749023
fit_T,0.951172,0.995117,0.984375,0.936523,0.749023
fit_T_prior,0.951172,0.995117,0.984375,0.936523,0.749023


Memorizing doc 2
Retrieving doc 2


,0,1,2,3,4
dot_T1,0.951172,0.995117,0.984375,0.939453,0.760742
bayes_T,0.948242,0.996094,0.983398,0.935547,0.749023
bayes_T_prior,0.948242,0.996094,0.983398,0.935547,0.749023
fit_T,0.948242,0.996094,0.983398,0.935547,0.749023
fit_T_prior,0.948242,0.996094,0.983398,0.935547,0.749023


Memorizing doc 3
Retrieving doc 3


,0,1,2,3,4
dot_T1,0.979492,0.994141,0.980469,0.937500,0.756836
bayes_T,0.971680,0.990234,0.976562,0.936523,0.748047
bayes_T_prior,0.971680,0.990234,0.976562,0.936523,0.748047
fit_T,0.971680,0.990234,0.976562,0.936523,0.748047
fit_T_prior,0.971680,0.990234,0.976562,0.936523,0.748047


Memorizing doc 4
Retrieving doc 4


,0,1,2,3,4
dot_T1,0.968750,0.990234,0.975586,0.938477,0.754883
bayes_T,0.961914,0.986328,0.969727,0.933594,0.747070
bayes_T_prior,0.961914,0.986328,0.969727,0.933594,0.747070
fit_T,0.961914,0.986328,0.969727,0.933594,0.747070
fit_T_prior,0.961914,0.986328,0.969727,0.933594,0.747070


Memorizing doc 5
Retrieving doc 5


,0,1,2,3,4
dot_T1,0.945312,0.982422,0.972656,0.926758,0.750000
bayes_T,0.940430,0.975586,0.971680,0.923828,0.743164
bayes_T_prior,0.940430,0.975586,0.971680,0.923828,0.743164
fit_T,0.940430,0.975586,0.971680,0.923828,0.743164
fit_T_prior,0.940430,0.975586,0.971680,0.923828,0.743164


In [36]:
# # Initializing the Memo Model from the configuration

# model = MeMoForCausalLM(config) 
# model.to('cuda')

# for i in range(5):
#     print(f"Memorizing doc {i+1}")
#     model.memorize_text(tok_data[i])
    
#     print(f"Retrieving doc {i+1}")  
#     pred = model.retrieve(tok_data[i]['input_ids']).lm_logits_unpacked.to("cpu")
#     for j in range(pred.shape[0]):
#         print(f'Score at level {j}: {(tok_data[i]["labels"] == pred[j].argmax(dim=-1)).sum().item() / len(tok_data[i]["labels"][0])}')

# model = model.to('cpu')
# del model
# torch.cuda.empty_cache()

In [37]:
# ((pred[-1] == 0).sum(axis=-1) == pred.shape[-1]).sum(-1)

In [38]:
model = MeMoForCausalLM(config) 
model.to('cuda')


for i in range(len(tok_data)):
    model.memorize_text(tok_data[i])
    print(f"-------------------------------- {i+1} docs memorized --------------------------------")
    for j in range(i+1):
        results = perform_evaluation(
            model=model,
            tokenizer=tokenizer,
            text=[data[j]['text']],
            starting_point=l**h,
            truncation=True
        )
        print(f"Checking doc {j+1}:")
        print(format_decoder_scores(results['decoder_scores']))

MeMoEmbedding circular_convolution_identity
MeMoEmbedding 0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  circular_convolution_identity
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  circular_convolution_identity
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
-------------------------------- 1 docs memorized --------------------------------
Checking doc 1:
decoder             nll       ppl  entropy   conf    top1    top5   top10  top100
bayes_T           0.000      1.00    0.000  1.000   1.000   1.000   1.000   1.000
bayes_T_prior     0.000      1.00    0.000  1.000   1.000   1.000   1.000   1.000
fit_T             0.000      1.00    0.000  1.000   1.000   1.000   1.000   1.000
fit_T_prior       0.000      1.00    0.000  1.000   1.000   1.

In [ ]:
exit()

: 